In [ ]:
"""Build the supplementary table of significant counts from the two summary text files.
Usage: python3 make_counts_table.py <channel_summary.txt> <region_summary.txt> <out.tex>
"""
import re, sys

SUB = {"sub01": "SUB1 (UWS)", "sub03": "SUB2 (MCS+)", "sub04": "SUB3 (MCS$-$)", "sub07": "SUB4 (MCS$-$)",
       "sub09": "SUB5 (MCS$-$)", "sub10": "SUB6 (MCS$-$)", "sub12": "SUB7 (MCS$-$)"}
COLS = ["delta", "theta", "alpha", "beta", "gamma", "Offse", "Expon"]
# section number in the summary files -> row-group title; the sign in the files is condition2 - condition1,
# which is the difference written in parentheses in each title
SECTIONS = [(2, r"Segmented pink noise vs.\ segmented music (difference: music $-$ pink noise)"),
            (3, r"Continuous pink noise vs.\ continuous music (difference: music $-$ pink noise)"),
            (4, r"Intermediate rest vs.\ segmented music (difference: music $-$ rest)"),
            (5, r"Intermediate rest vs.\ segmented pink noise (difference: pink noise $-$ rest)"),
            (6, r"Intermediate rest vs.\ continuous music (difference: music $-$ rest)")]

def parse(path):
    """-> {section: {"epochs": {sub: n}, "counts": {sub: {col: (n, plus, minus)}}}}"""
    out, sec = {}, None
    for line in open(path):
        m = re.match(r"=== (\d+)\.", line)
        if m:
            sec = int(m.group(1)); out[sec] = {"epochs": {}, "counts": {}}; continue
        if sec is None:
            continue
        if line.startswith("epochs per condition"):
            out[sec]["epochs"] = {s: int(n) for s, n in re.findall(r"'(sub\d+)': (\d+)", line)}; continue
        m = re.match(r"\s+(sub\d+)\s", line)
        if m:
            d = out[sec]["counts"].setdefault(m.group(1), {})
            for key, n, p, q in re.findall(r"(\w+):\s*(\d+)\(\+(\d+)/-(\d+)\)", line):
                n, p, q = int(n), int(p), int(q)
                assert n == p + q, (path, sec, m.group(1), key)      # internal consistency of the summary
                d[key] = (n, p, q)
    return out

def cell(v):
    n, p, q = v
    return "0" if n == 0 else f"{n} ({p}/{q})"

ch, rg = parse(sys.argv[1]), parse(sys.argv[2])
L = [r"\begingroup", r"\footnotesize", r"\setlength{\tabcolsep}{3.5pt}",
     r"\begin{xltabular}{\textwidth}{@{}l l r *{7}{>{\raggedleft\arraybackslash}X}@{}}",
     r"\caption{Number of significant channels (of 128) and regions (of 90) per subject, frequency band, and aperiodic "
     r"parameter, for the comparisons between music and pink noise and for the comparisons with the intermediate rest. "
     r"Each entry gives the number of significant units (FDR-corrected, $p < 0.05$). For entries above zero, the "
     r"parentheses give the number of units with a positive and with a negative difference in the epoch mean "
     r"(positive/negative). The direction of the difference is stated in each heading. $N$: number of epochs per "
     r"condition, identical for both levels. The corresponding counts for the initial rest versus continuous music "
     r"comparison are given in Tables~\ref{tab:split_half_channel} and~\ref{tab:split_half_region}.}",
     r"\label{tab:sig_counts_comparisons}\\",
     r"\toprule",
     r"\textbf{Subject} & \textbf{Level} & $N$ & $\delta$ & $\theta$ & $\alpha$ & $\beta$ & $\gamma$ & Offset & Exp. \\",
     r"\midrule", r"\endhead"]
for i, (sec, title) in enumerate(SECTIONS):
    assert ch[sec]["epochs"] == rg[sec]["epochs"], sec
    if i:
        L.append(r"\midrule")
    L.append(r"\multicolumn{10}{@{}l}{\textit{" + title + r"}} \\*")
    L.append(r"\addlinespace[2pt]")
    for s in sorted(ch[sec]["counts"]):
        n = ch[sec]["epochs"][s]
        for k, (lev, d) in enumerate((("Channel", ch), ("Region", rg))):
            row = [rf"\multirow{{2}}{{*}}{{{SUB[s]}}}" if k == 0 else "", lev, str(n) if k == 0 else ""]
            row += [cell(d[sec]["counts"][s][c]) for c in COLS]
            L.append(" & ".join(row) + (r" \\*" if k == 0 else r" \\"))
L += [r"\bottomrule", r"\end{xltabular}", r"\endgroup"]
open(sys.argv[3], "w").write("\n".join(L) + "\n")
print("rows:", sum(len(ch[s]["counts"]) for s, _ in SECTIONS) * 2)